In [35]:
from pydantic import BaseModel
from typing import List, Optional, Literal
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import ollama

In [37]:
class Violation(BaseModel):
    policy_id: str
    severity: Literal["Critical", "Warning"]
    explanation: str

In [39]:
# 1. We define what our "Tray" (State) is allowed to hold.
class ComplianceState(BaseModel):
    
    # This holds the raw text file the user uploaded. 
    # It must be a string (text).
    raw_document: str
    
    # This starts as an empty list []. 
    # Agent 1 (Retriever) will eventually fill this with relevant laws.
    retrieved_policies: List[str] = []
    
    # This starts as an empty list []. 
    # Agent 2 (Evaluator) will fill this with dictionaries of violations it finds.
    compliance_violations: List[Violation] = []
    
    # This starts as None (empty). 
    # Agent 3 (Writer) will fill this with the final text report at the very end.
    final_report: Optional[str] = None
    
    # This tracks where the tray is currently sitting in the shop.
    # We initialize it to start at the "Retrieval" step.
    current_step: str = "Retrieval"

In [9]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel


In [17]:
def retrieve_relevant_policies(state: ComplianceState):
    """
    Retrieves relevant policies from the vector store based on the query.
    """
    vector_db = Chroma(
        persist_directory="./chroma_langchain_db",
        collection_name="gdpr_collection",
        embedding_function=OllamaEmbeddings(model="nomic-embed-text")
        )
    
    relevant_docs = vector_db.similarity_search(state.raw_document, k=5)

    state.retrieved_policies = [doc.page_content for doc in relevant_docs]
    state.current_step = "Evaluation"

    return state


In [ ]:
def evaluate_compliance(state: ComplianceState):
    """Evaluate the compliance status of the given state."""

    system_instructions = """
        You are a compliance auditor. Your task is to analyze the provided [USER_DOCUMENT] against a set of [RETRIEVED_POLICIES].
        Output only this JSON format:
        {
            "status": "Compliant" | "Non-Compliant",
            "violations": [
                {
                "policy_id": "string",
                "severity": "Critical" | "Warning",
                "explanation": "string"
                }
            ]
        } 

        Do not include any introductory text or explanation outside of the JSON block.
        """
    
    user_input = f"""These are the legal/compliance requirements you must use for your evaluation: """

    policies_text = "#RETRIEVED_POLICIES# \n<policies>"

    for i, policy in enumerate(state.retrieved_policies):
        policies_text += f"Policy {i}: {policy}\n"

    policies_text += "</policies>"

    user_input += policies_text
    user_input += f"""This is the document you need to evaluate for compliance: #USER_DOCUMENT# \n{state.raw_document}"""

    message = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_input}
    ]

    payload = {
        "model": "llama3.2:3b",
        "messages": message,
        "stream": True
    }
    response = ollama.chat(model='llama3.1', messages=message, stream=True)

    return response



In [ ]:
app = FastAPI(title="Corporate Compliance Agent API", version="1.0")

In [18]:
compliance_state = ComplianceState(raw_document="Storing user data for 5 years in the database")
compliance_state = retrieve_relevant_policies(compliance_state)
evaluate_compliance(compliance_state)

GDPR Policies: ['in v olves the processing of personal data. It is also appropr iate to clar ify that data subjects continue t o enjo y all the \nOJ L, 12.7.2024\nEN\nELI: http://data.europa.eu/eli/reg/2024/1689/oj 3/144\n(\n7\n) Regulation (EC) No 765/2008 of the European Parl iament and of the Council of 9 July 2008 setting out the requirements f or \naccreditation and repealing Regulation (EEC) No 339/93 (OJ L 218, 13.8.2008, p. 30).\n(\n8', 'shall be accessible only to marke t sur veillance author ities and the Commission, unless the prospective provider or provid er \nhas given consent f or also making the inf or mation accessible the public.\n5. The EU database shall contain personal data only in so f ar as necessar y f or collecting and processing inf or mation in \naccordance with this Regulation. That inf or mation shall include the names and contact details of natural persons who are', '(a) recording of the per iod of each use of the syste m (star t date and time and end date

In [26]:
compliance_state.retrieved_policies

['in v olves the processing of personal data. It is also appropr iate to clar ify that data subjects continue t o enjo y all the \nOJ L, 12.7.2024\nEN\nELI: http://data.europa.eu/eli/reg/2024/1689/oj 3/144\n(\n7\n) Regulation (EC) No 765/2008 of the European Parl iament and of the Council of 9 July 2008 setting out the requirements f or \naccreditation and repealing Regulation (EEC) No 339/93 (OJ L 218, 13.8.2008, p. 30).\n(\n8',
 'shall be accessible only to marke t sur veillance author ities and the Commission, unless the prospective provider or provid er \nhas given consent f or also making the inf or mation accessible the public.\n5. The EU database shall contain personal data only in so f ar as necessar y f or collecting and processing inf or mation in \naccordance with this Regulation. That inf or mation shall include the names and contact details of natural persons who are',
 '(a) recording of the per iod of each use of the syste m (star t date and time and end date and time of 

In [27]:
# delimiter ="\n"

# delimiter.join(compliance_state.retrieved_policies)

for i, policy in enumerate(compliance_state.retrieved_policies):
    print(f"Policy {i}: {policy}")

Policy 0: in v olves the processing of personal data. It is also appropr iate to clar ify that data subjects continue t o enjo y all the 
OJ L, 12.7.2024
EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 3/144
(
7
) Regulation (EC) No 765/2008 of the European Parl iament and of the Council of 9 July 2008 setting out the requirements f or 
accreditation and repealing Regulation (EEC) No 339/93 (OJ L 218, 13.8.2008, p. 30).
(
8
Policy 1: shall be accessible only to marke t sur veillance author ities and the Commission, unless the prospective provider or provid er 
has given consent f or also making the inf or mation accessible the public.
5. The EU database shall contain personal data only in so f ar as necessar y f or collecting and processing inf or mation in 
accordance with this Regulation. That inf or mation shall include the names and contact details of natural persons who are
Policy 2: (a) recording of the per iod of each use of the syste m (star t date and time and end date and 